# 0

In [11]:
from datetime import time, datetime
import json
from moviepy.editor import VideoFileClip, concatenate_videoclips
import os
import pandas as pd
from pathlib import Path
import random
import subprocess
import sys
import tempfile
from typing import List, Tuple

curr_path = os.getcwd()
code_path = "E:/Documents/GitHub/RandSekai"
target_dir = r'E:\Videos\randsekai'

In [2]:
os.chdir(code_path)
from data_moe import df
df_vid = pd.read_excel('database/KaraSeka数据库.xlsx')
os.chdir(curr_path)

df = pd.merge(df, df_vid[['id', 'bv_off', 'page_off', 'start_t', 'end_t']], how='left', on='id')

['388', '2023/09/30-2023/10/12', '[[世界计划虚拟歌手演唱歌曲/收录歌曲#初音未来的激唱|{{lj|初音ミクの激唱}}]]', '[[初音未來]]', '{{color|red|200}}', '4:59', 'colspan="4" {{N/A|难度不存在}}', "'''{{color|#FF77DD|35}}'''", 'colspan="4" {{N/A|难度不存在}}', "'''{{color|#FF77DD|4500}}'''", ' 3周年限时开放']
['675', '2025/09/30-2025/10/06', '{{lj|プロセカULTIMATE楽曲メドレー}}', '[[初音未來]]・[[镜音铃·连]]・<br>[[巡音流歌]]・[[KAITO]]・<br>[[MEIKO]]', '162-{{color|red|300}}', '5:23', 'colspan="4" {{N/A|难度不存在}}', "'''{{color|#FF77DD|37}}'''", 'colspan="4" {{N/A|难度不存在}}', "'''{{color|#FF77DD|5197}}'''", " 5周年限时开放的高难度乐曲 BOSS RUSH第1弹，包含「{{lj|人生}}」、「{{lj|ダイジョブですか？}}」、「{{lj|おぎゃりないざー}}」、「What's up? Pop!」、「{{lj|メモリア}}」、「{{lj|ヤミナベ!!!!}}」6首歌曲串烧"]
['674', '2025/10/06-2025/10/11', '{{lj|MASTER高難易度楽曲メドレー}}', '[[初音未來]]・[[GUMI]]・<br>[[IA]]', '122-{{color|red|260}}', '5:15', 'colspan="4" {{N/A|难度不存在}}', "'''{{color|#884499|35}}'''", 'colspan="4" {{N/A|难度不存在}}', "'''{{color|#884499|3638}}'''", ' 5周年限时开放的高难度乐曲 BOSS RUSH第2弹，包含「{{lj|初音ミクの激唱}}」、「{{lj|六兆年と一夜物語}}」、「{{lj|腐れ外道とチョコレゐト}}」、「{

# 1
下载

In [3]:
df[df.bv_off.notnull()].head()

,id,date,title,vocal,bpm,duration,ez,nr,hd,ex,...,exn,maapdn,info,appdate,band,genre,bv_off,page_off,start_t,end_t
0,63,2020/09/30,needLe,初音未來,190,1:54,7,12,17,25,...,889,"(1029,)",在主线剧情中由Untitled化成,,ln,Leo/need演唱歌曲/原创歌曲,BV1KiySYBEHn,1,00:00:51,00:01:31
1,64,2020/10/18,ステラ,初音未來,197,2:13,8,13,17,26,...,955,"(1190, 1553)",第一期活动『[[世界计划 彩色舞台 feat. 初音未来/历史活动#1|{{lj|雨上がりの...,2023/09/30,ln,Leo/need演唱歌曲/原创歌曲,BV1KiySYBEHn,2,00:01:05,00:02:02
2,97,2021/01/10,霽れを待つ,初音未来,168,1:54,7,12,17,26,...,974,"(1106,)",第十期活动『[[世界计划 彩色舞台 feat. 初音未来/历史活动#10|{{lj|揺れるま...,,ln,Leo/need演唱歌曲/原创歌曲,BV1KizmBrEKG,4,00:00:55,00:01:39
3,132,2021/04/21,「1」,巡音流歌,145-148,1:38,8,13,16,23,...,619,"(814, 1001)",第二十期活动『[[世界计划 彩色舞台 feat. 初音未来/历史活动#20|Resonate...,2025/01/31,ln,Leo/need演唱歌曲/原创歌曲,BV1KiySYBEHn,3,00:00:44,00:01:11
4,130,2021/07/07,フロムトーキョー,初音未來,130,1:39,6,12,18,24,...,640,"(814,)",第二十七期活动『[[世界计划 彩色舞台 feat. 初音未来/历史活动#27|Unnamed...,,ln,Leo/need演唱歌曲/原创歌曲,BV1ExWwz2ERj,2,00:00:41,00:01:04


In [4]:
for idx, row in df[df.bv_off.notnull()].iterrows():
    file_id = str(row['id'])
    output_file = Path(target_dir) / f"{file_id}.mp4"

    if output_file.exists():
#         print(f"{file_id} already exists")
        continue

#     print(f"Downloading {file_id}")
    cmd = ["bbdown", row['bv_off'], "-p", str(row['page_off']), "--work-dir", target_dir,
           "-F", file_id, "-M", file_id] # bbdown BV1KiySYBEHn -p 2 --work-dir 'E:\Videos\randsekai' -F '64' -M '64'
    try:
        # 运行命令，不启用 shell（更安全）
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"Successed ({file_id})")
#         print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Failed ({file_id}): {e}")
#         print(e.stderr)

# 2
抽样

In [12]:
def parse_time_to_seconds(time_val) -> float:
    """
    将各种时间格式转换为秒数（浮点数）。
    支持：
    - 数字（int/float）：直接作为秒数
    - 字符串 "HH:MM:SS" 或 "HH:MM:SS.mmm"
    - 字符串 "MM:SS" 或 "SS"
    - datetime.time 对象
    - datetime.datetime 对象（提取时间部分）
    """
    if isinstance(time_val, (int, float)):
        return float(time_val)
    if isinstance(time_val, str):
        parts = time_val.strip().split(':')
        if len(parts) == 3:  # HH:MM:SS
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
        elif len(parts) == 2:  # MM:SS
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 1:  # SS
            return float(parts[0])
        else:
            raise ValueError(f"不支持的时间格式: {time_val}")
    if isinstance(time_val, time):
        # datetime.time 对象
        return time_val.hour * 3600 + time_val.minute * 60 + time_val.second + time_val.microsecond / 1e6
    if isinstance(time_val, datetime):
        # datetime.datetime 对象，提取时间部分
        t = time_val.time()
        return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
    raise TypeError(f"不支持的时间类型: {type(time_val)}")

def random_sample_ids_until_duration(
    df: pd.DataFrame,
    target_duration_sec: float = 3600.0,
    seed: int = None
) -> Tuple[List[str], float]:
    """
    从 df 中不放回地随机抽取 id，累计时长，直到达到 target_duration_sec
    
    参数:
        df: 必须包含列 'id', 'start_t', 'end_t'
        target_duration_sec: 目标总时长（秒）
        seed: 随机种子，用于复现
    
    返回: 抽中的id列表（按抽取顺序），以及实际累计时长
    """
    if seed is not None:
        random.seed(seed)

    # 复制数据，避免修改原df
    df_work = df.copy()
    
    # 将 start_time 和 end_time 转换为秒数
    df_work['start_sec'] = df_work['start_t'].apply(parse_time_to_seconds)
    df_work['end_sec'] = df_work['end_t'].apply(parse_time_to_seconds)
    df_work['duration_sec'] = df_work['end_sec'] - df_work['start_sec']
    
    # 过滤掉时长 <= 0 的记录（避免无效片段）
    df_work = df_work[df_work['duration_sec'] > 0].copy()
    if df_work.empty:
        raise ValueError("没有有效的片段（时长 > 0）")
    
    # 打乱顺序
    df_shuffled = df_work.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    selected_ids = []
    total_duration = 0.0
    
    for _, row in df_shuffled.iterrows():
        if total_duration >= target_duration_sec:
            break
        # 加入当前片段
        selected_ids.append(row['id'])
        total_duration += row['duration_sec']
    
    return selected_ids, total_duration

In [13]:
id_order, total_t = random_sample_ids_until_duration(df[df.bv_off.notnull()], target_duration_sec=150, seed=42)
print(f"抽中 id 顺序: {id_order}")
print(f"累计时长: {total_t:.2f} 秒 ({total_t/60:.2f} 分钟)")

抽中 id 顺序: ['64', '130', '97', '63']
累计时长: 164.00 秒 (2.73 分钟)


# 3
剪辑

In [6]:
def get_video_resolution(file_path):
    """使用 ffprobe 获取视频宽高"""
    cmd = [
        'ffprobe', '-v', 'error', '-select_streams', 'v:0',
        '-show_entries', 'stream=width,height', '-of', 'json',
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='ignore', check=True)
    info = json.loads(result.stdout)
    width = info['streams'][0]['width']
    height = info['streams'][0]['height']
    return width, height

def cut_segment(input_file, start, end, output_path, target_width=1920, target_height=1080, reencode=True):
    """
    使用 FFmpeg 裁剪视频片段，并强制缩放到目标分辨率（如果需要）
    """
    w, h = get_video_resolution(input_file)
    need_scale = (w != target_width or h != target_height)

    cmd = ['ffmpeg', '-y', '-i', input_file, '-ss', str(start), '-to', str(end)]

    if need_scale:
        vf = f'scale={target_width}:{target_height}:force_original_aspect_ratio=decrease,pad={target_width}:{target_height}:(ow-iw)/2:(oh-ih)/2'
        cmd += ['-vf', vf]

    if reencode:
        cmd += ['-c:v', 'libx264', '-c:a', 'aac', '-preset', 'fast', '-vsync', '1', '-async', '1']
    else:
        cmd += ['-c', 'copy', '-avoid_negative_ts', 'make_zero']

    cmd.append(output_path)

#     print("cut_segment", cmd)
    subprocess.run(cmd, check=True, capture_output=True, text=True, encoding='utf-8', errors='ignore')
    return output_path

def concat_videos_by_ids(df, id_list, output_path, input_path='', start_col='start_t', end_col='end_t',
                         reencode=True, temp_dir=None):
    """
    根据 id 顺序截取视频片段，强制缩放到 1920x1080，然后用 MoviePy 拼接
    """
    # 统一 id 类型
    df = df.copy()
    df['id'] = df['id'].astype(str)
    id_list = [str(i) for i in id_list]

    # 创建临时目录
    if temp_dir is None:
        temp_dir = tempfile.mkdtemp()
    else:
        Path(temp_dir).mkdir(parents=True, exist_ok=True)

    segment_paths = []
    try:
        # 步骤1: 使用 FFmpeg 裁剪每个片段（并缩放）
        for idx, vid_id in enumerate(id_list):
            row = df[df['id'] == vid_id]
            if row.empty:
                raise ValueError(f"ID '{vid_id}' 不存在")
            row = row.iloc[0]
            input_file = os.path.join(input_path, vid_id+'.mp4')
            start = row[start_col]
            end = row[end_col]

            seg_path = os.path.join(temp_dir, f"segment_{idx:03d}.mp4")
            cut_segment(input_file, start, end, seg_path, reencode=reencode)
            print(vid_id+' finished...')
            segment_paths.append(seg_path)

        if not segment_paths:
            raise RuntimeError("没有生成任何片段")

        # 步骤2: 使用 MoviePy 拼接
        clips = [VideoFileClip(p) for p in segment_paths]
        # method="compose" 可以处理不同尺寸帧率的片段，但做完预处理了默认也够用
        final_clip = concatenate_videoclips(clips, method="compose")
        final_clip.write_videofile(output_path, codec='libx264', audio_codec='aac', temp_audiofile=None, remove_temp=True)
        # 关闭 clips 释放资源
        for clip in clips:
            clip.close()
        final_clip.close()

        print(f"Success: {output_path}")

    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")
        raise
    finally:
        # 清理临时文件
        import shutil
        shutil.rmtree(temp_dir, ignore_errors=True)

In [7]:
concat_videos_by_ids(df, id_order, 'output_combined.mp4', input_path=target_dir, reencode=True)

97 finished...
63 finished...
132 finished...
64 finished...
130 finished...
Moviepy - Building video output_combined.mp4.
MoviePy - Writing audio in output_combinedTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video output_combined.mp4



Moviepy - Done !
Moviepy - video ready output_combined.mp4
Success: output_combined.mp4
